In [ ]:
#| default_exp display

# display
> display helpers

In [ ]:
#| export
from typing import Mapping, Sequence
from fasthtml.common import show
from fasthtml.components import Div, Details, Summary, Ul, Li, Span
from fasthtml.xtend import Style
from dialoghelper.core import add_html
from pote.common import is_listy, shorten

In [ ]:
import json
from functools import partial
from pathlib import Path
from typing import cast

from fastcore.test import *
from fastcore.xml import FT
from fastcore.xml import to_xml

In [ ]:
from typing import Any
from typing import Literal

from pote.common import *

In [ ]:
#| export
_n = '\n'

## DetailsJSON
> Collapsible JSON/dict viewer component for notebooks

A FastHTML component that renders dictionaries and nested data as expandable `<details>` elements with syntax highlighting.

In [ ]:
%%HTML
<style>
    details.json ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0px; }
</style>
<details open class="json">
<summary>Apollo astronauts</summary>
<ul>
  <li><span>1</span>: Neil Armstrong</li>
  <li><span>2</span>: Alan Bean</li>
  <li><details>
<summary>Apollo 11</summary>
<ul>
  <li><span>1</span>: Neil Armstrong</li>
  <li><span>2</span>: Alan Bean</li>
  <li><div><span>3</span>: Buzz Aldrin</div></li>
  <li><span>4</span>: Edgar Mitchell</li>
  <li><span>5</span>: Alan Shepard</li>
</ul></li>
  <li><span>4</span>: Edgar Mitchell</li>
  <li><span>5</span>: Alan Shepard</li>
</ul>

</details>

HTML(<style>
    details.json ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0px; }
</style>
<details open class="json">
<summary>Apollo astronauts</summary>
<ul>
  <li><span>1</span>: Neil Armstrong</li>
  <li><span>2</span>: Alan Bean</li>
  <li><details>
<summary>Apollo 11</summary>
<ul>
  <li><span>1</span>: Neil Armstrong</li>
  <li><span>2</span>: Alan Bean</li>
  <li><div><span>3</span>: Buzz Aldrin</div></li>
  <li><span>4</span>: Edgar Mitchell</li>
  <li><span>5</span>: Alan Shepard</li>
</ul></li>
  <li><span>4</span>: Edgar Mitchell</li>
  <li><span>5</span>: Alan Shepard</li>
</ul>

</details>
)

In [ ]:
#| export
def Val(v): 
    "Render value with appropriate CSS class based on type"
    c = (
        'null' if v is None else 
        'true' if v is True else 
        'false' if v is False else 
        'string' if isinstance(v, str) else 
        'number' if isinstance(v, (int, float)) else 
        '')
    return Span(shorten(v, 'r', 140) if v is not None else 'None', cls=f"v {c}")

In [ ]:
#| export
def NameVal(k, v):
    "Render key-value pair with name and value styling"
    return Span(Span(k, cls='n'), ': ', Val(v))

In [ ]:
#| export
class DetailsJSON(dict):
    "Interactive collapsible JSON viewer with HTML details/summary structure"
    def __init__(self, *args, summary:str='', open:bool=True, openall:bool=False, skip:Sequence[str]=(), **kwargs):
        super().__init__(*args, **kwargs)
        self.summary, self.open, self.openall, self.skip = str(summary), open, openall, skip
    def show(self): show(self)
    def __ft__(self, d:Mapping|None=None, summary:str|None=None, lvl:int=0, open:bool=False):
        if d is None: d = self; summary = self.summary or 'summary'; open=self.open
        open = self.openall or open
        return (
            # Style(self._css_) if lvl == 0 else (), 
            Details(open=open, cls="json")(
                Summary(summary, _n),
                Ul()(*(
                    Li(NameVal(k, v)) if k in self.skip else
                    self.__ft__(v, k, lvl+1) if isinstance(v, Mapping) else
                    self.__ft__(dict(list(zip(range(len(v)), v))), k, lvl+1) if is_listy(v) else
                    Li(NameVal(k, v)) 
                    for k,v in d.items()))))
    # _css_ = 'details ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0; } '
    _css_ = (
        'details.json ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0; } '
        '''details.json .string { color: #24837b; } details.json .string::before { content: "'"; } details.json .string::after { content: "'"; } '''
        'details.json .number { color: #ad8301; } '
        'details.json .true { color: blue; } '
        'details.json .false { color: red; } '
        'details.json .null { color: gray; } '
        'details.json span.n { color: darkgrey; } '
    )
    @classmethod
    def setup_style(cls):
        _stl = Div(Style(cls._css_), hx_swap_oob='beforeend:#dialog-container')
        add_html(_stl)

In [ ]:
#| export
DetailsJSON.setup_style()

In [ ]:
dtl = DetailsJSON({
    '1': 'Neil Armstrong',
    '2': 'Alan Bean',
    '3': 'Buzz Aldrin',
    'letters': {
        'a': 1, 
        'b': 2
        },
    '5': 'Edgar Mitchell',
    '6': 'Alan Shepard'
}, summary='Apollo astronauts')

test_eq(val_at(dtl, '5'), 'Edgar Mitchell')
test_eq(val_at(dtl, 'letters.a'), 1)
print(to_xml(dtl))
dtl.show()

<details open class="json"><summary>Apollo astronauts
</summary>  <ul>
    <li>
<span><span class="n">1</span>: <span class="v string">Neil Armstrong</span></span>    </li>
    <li>
<span><span class="n">2</span>: <span class="v string">Alan Bean</span></span>    </li>
    <li>
<span><span class="n">3</span>: <span class="v string">Buzz Aldrin</span></span>    </li>
<details class="json"><summary>letters
</summary>      <ul>
        <li>
<span><span class="n">a</span>: <span class="v number">1</span></span>        </li>
        <li>
<span><span class="n">b</span>: <span class="v number">2</span></span>        </li>
      </ul>
</details>    <li>
<span><span class="n">5</span>: <span class="v string">Edgar Mitchell</span></span>    </li>
    <li>
<span><span class="n">6</span>: <span class="v string">Alan Shepard</span></span>    </li>
  </ul>
</details>


HTML(<details open class="json"><summary>Apollo astronauts
</summary>  <ul>
    <li>
<span><span class="n">1</span>: <span class="v string">Neil Armstrong</span></span>    </li>
    <li>
<span><span class="n">2</span>: <span class="v string">Alan Bean</span></span>    </li>
    <li>
<span><span class="n">3</span>: <span class="v string">Buzz Aldrin</span></span>    </li>
<details class="json"><summary>letters
</summary>      <ul>
        <li>
<span><span class="n">a</span>: <span class="v number">1</span></span>        </li>
        <li>
<span><span class="n">b</span>: <span class="v number">2</span></span>        </li>
      </ul>
</details>    <li>
<span><span class="n">5</span>: <span class="v string">Edgar Mitchell</span></span>    </li>
    <li>
<span><span class="n">6</span>: <span class="v string">Alan Shepard</span></span>    </li>
  </ul>
</details>)

In [ ]:
d= {
    "idx": 1,
    "cell_type": "code",
    "source": "# cell 1\nprint('hello')",
    "id": "W1sZmlsZQ==",
    "metadata": {
        "brd": {
            "id": "717322f8-95fa-425c-839d-8b9e7d4ef921"
        }
    },
    "outputs": [
        {'output_type': 'stream', 'name': 'stdout', 'text': '1\n'},
        {'output_type': 'stream', 'name': 'stdout', 'text': '2\n'}
    ],
    "execution_count": 1
}
DetailsJSON(d, openall=True).show()

HTML(<details open class="json"><summary>summary
</summary>  <ul>
    <li>
<span><span class="n">idx</span>: <span class="v number">1</span></span>    </li>
    <li>
<span><span class="n">cell_type</span>: <span class="v string">code</span></span>    </li>
    <li>
<span><span class="n">source</span>: <span class="v string"># cell 1
print('hello')</span></span>    </li>
    <li>
<span><span class="n">id</span>: <span class="v string">W1sZmlsZQ==</span></span>    </li>
<details open class="json"><summary>metadata
</summary>      <ul>
<details open class="json"><summary>brd
</summary>          <ul>
            <li>
<span><span class="n">id</span>: <span class="v string">717322f8-95fa-425c-839d-8b9e7d4ef921</span></span>            </li>
          </ul>
</details>      </ul>
</details><details open class="json"><summary>outputs
</summary>      <ul>
<details open class="json"><summary>0
</summary>          <ul>
            <li>
<span><span class="n">output_type</span>: <span class="v string">stream</span></span>            </li>
            <li>
<span><span class="n">name</span>: <span class="v string">stdout</span></span>            </li>
            <li>
<span><span class="n">text</span>: <span class="v string">1
</span></span>            </li>
          </ul>
</details><details open class="json"><summary>1
</summary>          <ul>
            <li>
<span><span class="n">output_type</span>: <span class="v string">stream</span></span>            </li>
            <li>
<span><span class="n">name</span>: <span class="v string">stdout</span></span>            </li>
            <li>
<span><span class="n">text</span>: <span class="v string">2
</span></span>            </li>
          </ul>
</details>      </ul>
</details>    <li>
<span><span class="n">execution_count</span>: <span class="v number">1</span></span>    </li>
  </ul>
</details>)

In [ ]:
request = {
    'headers': {
        'HX-Request': 'true',
        'HX-Current-URL': 'vscode-webview://1ql27...enderer'
    },
    'headerNames': {
        'hx-request': 'HX-Request',
        'hx-current-url': 'HX-Current-URL'
    },
    'status': 0,
    'method': 'GET',
    'url': '/DetailsJSON_5096628128/Apollo 11/Buzz Aldrin',
    'async': True,
    'timeout': 0,
    'withCredentials': False,
    'body': None,
    'req_id': '68ffadb0-958d-4346-b314-d9d62ca247d7'
}

response = {
    'headers': {
        'content-length': '441',
        'content-type': 'text/html; charset=utf-8',
        'last-modified': 'Fri, 15 Nov 2024 16:22:15 GMT',
        'cache-control': 'no-store, no-cache, must-revalidate'
    },
    'status': 200,
    'statusText': 'OK',
    'data': '<details open><summary>Buzz Aldrin</summary>\n   <ul>\n     '
'<li>Pilot on Gemini 12 and Lunar Module pilot on Apollo 11.</li>\n     '
'<li>Aldrin was the second person to walk on the moon.</li>\n     <li>The maiden '
'name of Aldrin&#x27;s mother was &quot;Moon.&quot;</li>\n     <li>While Neil was '
'the first human to step onto the moon, I&#x27;m the first alien from another '
'world to enter a spacecraft that was going to Earth.</li>\n   '
'</ul>\n</details>',
    'xml': None,
    'finalUrl': 'http://nb/DetailsJSON_5096628128/Apollo%2011/Buzz%20Aldrin',
    'req_id': '68ffadb0-958d-4346-b314-d9d62ca247d7'
}

In [ ]:
req = DetailsJSON(request, summary='request', open='all')
req.show()

HTML(<details open="all" class="json"><summary>request
</summary>  <ul>
<details class="json"><summary>headers
</summary>      <ul>
        <li>
<span><span class="n">HX-Request</span>: <span class="v string">true</span></span>        </li>
        <li>
<span><span class="n">HX-Current-URL</span>: <span class="v string">vscode-webview://1ql27...enderer</span></span>        </li>
      </ul>
</details><details class="json"><summary>headerNames
</summary>      <ul>
        <li>
<span><span class="n">hx-request</span>: <span class="v string">HX-Request</span></span>        </li>
        <li>
<span><span class="n">hx-current-url</span>: <span class="v string">HX-Current-URL</span></span>        </li>
      </ul>
</details>    <li>
<span><span class="n">status</span>: <span class="v number">0</span></span>    </li>
    <li>
<span><span class="n">method</span>: <span class="v string">GET</span></span>    </li>
    <li>
<span><span class="n">url</span>: <span class="v string">/DetailsJSON_5096628128/Apollo 11/Buzz Aldrin</span></span>    </li>
    <li>
<span><span class="n">async</span>: <span class="v true">True</span></span>    </li>
    <li>
<span><span class="n">timeout</span>: <span class="v number">0</span></span>    </li>
    <li>
<span><span class="n">withCredentials</span>: <span class="v false">False</span></span>    </li>
    <li>
<span><span class="n">body</span>: <span class="v null">None</span></span>    </li>
    <li>
<span><span class="n">req_id</span>: <span class="v string">68ffadb0-958d-4346-b314-d9d62ca247d7</span></span>    </li>
  </ul>
</details>)

In [ ]:
resp = DetailsJSON(response, summary='response')
resp.show()

HTML(<details open class="json"><summary>response
</summary>  <ul>
<details class="json"><summary>headers
</summary>      <ul>
        <li>
<span><span class="n">content-length</span>: <span class="v string">441</span></span>        </li>
        <li>
<span><span class="n">content-type</span>: <span class="v string">text/html; charset=utf-8</span></span>        </li>
        <li>
<span><span class="n">last-modified</span>: <span class="v string">Fri, 15 Nov 2024 16:22:15 GMT</span></span>        </li>
        <li>
<span><span class="n">cache-control</span>: <span class="v string">no-store, no-cache, must-revalidate</span></span>        </li>
      </ul>
</details>    <li>
<span><span class="n">status</span>: <span class="v number">200</span></span>    </li>
    <li>
<span><span class="n">statusText</span>: <span class="v string">OK</span></span>    </li>
    <li>
<span><span class="n">data</span>: <span class="v string">&lt;details open&gt;&lt;summary&gt;Buzz Aldrin&lt;/summary&gt;
   &lt;ul&gt;
     &lt;li&gt;Pilot on Gemini 12 and Lunar Module pilot on Apollo 11.&lt;/li&gt;
     &lt;li&gt;Aldrin w…</span></span>    </li>
    <li>
<span><span class="n">xml</span>: <span class="v null">None</span></span>    </li>
    <li>
<span><span class="n">finalUrl</span>: <span class="v string">http://nb/DetailsJSON_5096628128/Apollo%2011/Buzz%20Aldrin</span></span>    </li>
    <li>
<span><span class="n">req_id</span>: <span class="v string">68ffadb0-958d-4346-b314-d9d62ca247d7</span></span>    </li>
  </ul>
</details>)

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dutil import dlg_export
# dlg_export()